# Lab 15/16 — Speculative decoding, measured

Build the speculative-decoding rule yourself — first on **dummy distributions** (where correctness is
provable), then as a **real two-model stitch** (gpt2 verified against distilgpt2, a same-tokenizer pair).
Read the [lab brief](Lab15_16.md) first. Marked on **relationships between your own numbers**; the
sampling parts are seeded from your roll number, so your numbers are yours.

**Before you run anything:** set your identity below. Fill every 📝 cell. Run top to bottom, then run the
final export cell and submit the two files it names. CPU is enough (`gpt2` + `distilgpt2`).

In [ ]:
ROLL_NUMBER = ""      # <- your roll number, e.g. "202512345"
NAME        = ""      # <- your name

assert ROLL_NUMBER and NAME, "Set ROLL_NUMBER and NAME before running the rest."

## Setup — dummy data + the model pair (given)

In [ ]:
import json, platform, sys, hashlib
from pathlib import Path
import numpy as np
import torch, torch.nn.functional as F
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

SEED = int(hashlib.sha256(ROLL_NUMBER.encode()).hexdigest(), 16) % (2**31)   # your seed
rng = np.random.default_rng(SEED)

# --- dummy distributions (Parts 1-3): 5-symbol toy, one row per drafted position (+ bonus row) ---
VOCAB = ["cat", "sat", "on", "the", "mat"]; V = len(VOCAB); k = 3
p = np.array([[0.50,0.20,0.10,0.10,0.10],
              [0.10,0.60,0.10,0.10,0.10],
              [0.10,0.10,0.20,0.50,0.10],
              [0.20,0.20,0.20,0.20,0.20]])   # (k+1, V) target; row k = bonus
q = np.array([[0.70,0.10,0.10,0.05,0.05],
              [0.10,0.50,0.20,0.10,0.10],
              [0.10,0.10,0.60,0.10,0.10]])   # (k, V) draft (over-proposes cat, on)
drafts = [0, 1, 2]
assert np.allclose(p.sum(1),1) and np.allclose(q.sum(1),1)

# --- the real model pair (Part 4): same tokenizer, both small, CPU-fast ---
TARGET, DRAFT = "gpt2", "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(TARGET)
target = AutoModelForCausalLM.from_pretrained(TARGET).eval()
draft  = AutoModelForCausalLM.from_pretrained(DRAFT).eval()
assert target.config.vocab_size == draft.config.vocab_size, "the stitch needs a shared vocabulary"

RESULTS = {}
print("dummy vocab", V, "| models", TARGET, "<-", DRAFT, "| shared vocab", target.config.vocab_size, "| seed", SEED)

---
## Part 1 — The acceptance branch (dummy)

Accept a drafted token `x` at position `i` with probability `min(1, p[i,x] / q[i,x])`.

📝 **Predict.** When is the accept probability exactly **1**? Position 0 drafted `"cat"` with `q=0.70`,
`p=0.50` — what is its accept probability, and why does over-proposing get **throttled**?

*(your predictions here)*

In [ ]:
def is_accepted(p_i, q_i, x, u):
    """Ratio test for ONE drafted token. u is a uniform(0,1) draw passed in for reproducibility.
    Accept with probability min(1, p_i[x] / q_i[x]); return True/False."""
    # TODO: your code here
    raise NotImplementedError


# harness (given): empirical accept rate per position should track min(1, p/q)
emp = [float(np.mean([is_accepted(p[i], q[i], drafts[i], u) for u in rng.random(20000)])) for i in range(k)]
theo = [float(min(1.0, p[i, drafts[i]]/q[i, drafts[i]])) for i in range(k)]
RESULTS["accept"] = dict(positions=list(range(k)), empirical=[round(x,3) for x in emp],
                         theoretical=[round(x,3) for x in theo])
for i in range(k): print(f"pos {i} ({VOCAB[drafts[i]]}): empirical {emp[i]:.3f}  vs  min(1,p/q) {theo[i]:.3f}")

📝 **Explain.** Why is the accept probability capped at 1? What does over-proposing (`q > p`) do to
the long-run accept rate, and why is that the *right* correction?

*(your explanation here)*

---
## Part 2 — Rejection sampling: the residual (dummy)

A reject resamples from the **residual** `norm(max(0, p − q))`, **not** from `p`. That correction is what
makes the emitted stream exactly `p` — and a version that resamples from `p` has the *same acceptance rate*
but a silently wrong output.

📝 **Predict.** Over many single-step emissions, what distribution should the first emitted token follow?
What will the **broken** version (resample from `p`) get wrong that the acceptance rate cannot reveal?

*(your predictions here)*

In [ ]:
def residual(p_i, q_i):
    """Distribution to sample from after a rejection: proportional to max(0, p_i - q_i), renormalised."""
    # TODO: your code here
    raise NotImplementedError

def sample_on_reject(p_i, q_i):
    """Draw a replacement token id from residual(p_i, q_i) using rng."""
    # TODO: your code here (use residual + rng.choice)
    raise NotImplementedError


# reference single-position emitter (given): draft x ~ q0, accept/reject, emit ~ p0
def emit_once(sampler):
    x = int(rng.choice(V, p=q[0]))
    if is_accepted(p[0], q[0], x, rng.random()):
        return x
    return sampler(p[0], q[0])              # rejected -> replacement from the residual

def broken_sampler(p_i, q_i):               # the silent bug: resample from p, not the residual
    return int(rng.choice(V, p=p_i))

def dist_of(sampler, n=60000):
    c = np.zeros(V)
    for _ in range(n): c[emit_once(sampler)] += 1
    return c / c.sum()

emp_p   = dist_of(sample_on_reject)
brk_p   = dist_of(broken_sampler)
tv = lambda a, b: 0.5*float(np.abs(a-b).sum())
RESULTS["emit"]   = dict(target_p0=[round(x,3) for x in p[0]], empirical=[round(x,3) for x in emp_p],
                         tv_distance=round(tv(emp_p, p[0]), 4))
RESULTS["broken"] = dict(tv_distance=round(tv(brk_p, p[0]), 4))
print("target p0        :", np.round(p[0],3))
print("correct emitted  :", np.round(emp_p,3), " TV =", RESULTS["emit"]["tv_distance"])
print("broken  emitted  :", np.round(brk_p,3), " TV =", RESULTS["broken"]["tv_distance"], " <- should be far larger")

📝 **Explain (the paragraph that carries this part).** Why the residual and not `p`? An implementation
that resamples from `p` has an *identical acceptance rate* — name what it gets wrong and why only a
distribution test (not a latency or acceptance dashboard) catches it.

*(your answer here)*

---
## Part 3 — Vectorised verification (dummy)

Verification is **one pass over k+1 positions**, so the `k` ratio tests should be array ops, not a python
loop; only the *cut* at the first reject stays sequential.

📝 **Predict.** How do you turn a boolean accept-vector into the **first-reject index** in one step? Which
part must stay sequential, and how does that mirror "verification is a prefill"?

*(your predictions here)*

In [ ]:
def verify_vectorised(p, q, drafts, u):
    """Verify all k drafts at once. p:(k+1,V) q:(k,V) drafts:(k,) u:(k,) uniforms.
    Return (n_accepted, next_token): n_accepted in 0..k; next_token is a residual draw at the
    cut, or a bonus draw from p[k] if all k accepted. Vectorise the TEST; the cut is sequential."""
    # TODO: your code here
    raise NotImplementedError


# harness (given): must agree with a scalar reference built on your Part 1-2 functions
def verify_scalar(p, q, drafts, u):
    for i in range(k):
        if u[i] < min(1.0, p[i, drafts[i]]/q[i, drafts[i]]): continue
        return i, sample_on_reject(p[i], q[i])
    return k, int(rng.choice(V, p=p[k]))

agree = 0; N = 3000
for _ in range(N):
    u = rng.random(k)
    na_v, _ = verify_vectorised(p, q, drafts, u)
    na_s, _ = verify_scalar(p, q, drafts, u)   # compare the CUT (n_accepted); the token draw is random
    agree += (na_v == na_s)
RESULTS["vector"] = dict(agreement=round(agree/N, 4), n_checks=N)
print(f"vectorised vs scalar cut agreement: {agree/N:.4f}  ({N} checks)")

📝 **Explain.** What is embarrassingly parallel here and what is not? Tie it to Lecture 15/16's
"verification is a prefill".

*(your explanation here)*

---
## Part 4 — The real stitch: gpt2 verified against distilgpt2

Now the same rule on real models, in the **greedy** case (`T=0`), where correctness is *exactly* checkable:
speculative-greedy output must equal plain-greedy output, token for token — in **fewer target passes**.

Greedy collapses the accept test to an equality: accept the draft iff it equals the target's argmax.

📝 **Predict.** Will the two token streams be identical? Will speculative decoding use fewer **target
forward passes** than tokens generated? Why is the block yield bounded in `[1, k+1]`?

*(your predictions here)*

In [ ]:
PROMPT, N_TOKENS, K = "The capital of France is", 24, 4
ids0 = tokenizer(PROMPT, return_tensors="pt").input_ids

def plain_greedy(ids, n):                    # reference (given): n target passes, n tokens
    ids = ids.clone()
    for _ in range(n):
        with torch.no_grad(): nt = target(ids).logits[0,-1].argmax()
        ids = torch.cat([ids, nt.view(1,1)], 1)
    return ids[0, -n:].tolist()

def speculative_greedy(ids, n, k):
    """Greedy speculative decode. Return (tokens, target_passes, accept_lengths):
       - draft k tokens autoregressively from `draft` (argmax each step)
       - ONE `target` forward pass over the k drafts -> logits at each position
       - accept while target's argmax == the draft; at the first mismatch emit target's argmax and stop;
         if all k accepted, emit the bonus (target's argmax at position k)
       - count ONE target pass per block; record the accepted-run length per block
    """
    # TODO: your code here
    raise NotImplementedError


plain = plain_greedy(ids0, N_TOKENS)
spec, passes, acc_lens = speculative_greedy(ids0, N_TOKENS, K)
RESULTS["stitch"] = dict(target=TARGET, draft=DRAFT, k=K, n_tokens=N_TOKENS,
                         identical_to_greedy=bool(spec[:N_TOKENS] == plain),
                         target_passes=int(passes),
                         accept_lengths=[int(a) for a in acc_lens])
print("identical to plain greedy:", RESULTS["stitch"]["identical_to_greedy"])
print(f"target passes: {passes} for {N_TOKENS} tokens  (plain greedy would use {N_TOKENS})")
print("accepted-run lengths per block:", acc_lens)

📝 **Explain (the write-up).** Acceptance ≠ speedup: on this CPU, is speculative decoding actually
*faster* in wall-clock than plain greedy? Fewer target passes, yet the drafter runs its own passes —
reconcile the two. And: why does a bad drafter cost speed but **never** accuracy?

*(your answer here)*

---
## Submit

Run this last. It writes `submission_lab15_16_<roll>.json`. Submit **two files**: that JSON and this
executed notebook with every 📝 cell filled.

In [ ]:
env = dict(platform=platform.platform(), python=sys.version.split()[0],
           torch=torch.__version__, transformers=transformers.__version__,
           numpy=np.__version__, seed=SEED, target=TARGET, draft=DRAFT,
           linux=sys.platform.startswith("linux"))
sub = dict(roll=ROLL_NUMBER, name=NAME, env=env, results=RESULTS)
out = Path(f"submission_lab15_16_{ROLL_NUMBER}.json")
out.write_text(json.dumps(sub, indent=2))
print("wrote", out, "| parts:", list(RESULTS))
assert set(RESULTS) >= {"accept","emit","broken","vector","stitch"}, "run every part before exporting"